# Build an SRE Incident Response Agent with Claude Managed Agents

## Introduction

When a production alert fires at 3 a.m., someone has to pull the logs, find the right runbook, trace the misconfiguration, open a PR, and get it approved. An agent can take that first pass for you and have a fix waiting for review by the time you're at the keyboard — as long as it has the right context and an authenticated, authorized human makes the final call.

[Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview) gives you the scalable infrastructure, sandboxing, & security pieces to build that with ease. In this tutorial you'll wire them together:

- A simulated **PagerDuty webhook** triggers your Claude Managed Agent with one API call.
- A **Skill** teaches the agent your team's runbook conventions, so it knows where to look.
- The built-in `bash`/`read`/`edit` tools let it investigate logs and infrastructure code in a sandbox.
- **Custom tools** let it open a pull request and ask a human to approve before merging — your code handles those calls, so you decide what "open a PR" actually does.
- The **Anthropic Console** records every step automatically, providing you complete observability.

Everything below runs with only `ANTHROPIC_API_KEY`. PagerDuty, GitHub, and Datadog are mocked with local fixtures so you can focus on the Managed Agents pieces; the closing section shows how to swap each mock for the real service.

### What you'll learn

- Upload a Skill and attach it to a Claude Managed Agent
- Mix the built-in toolset with custom tools your application handles
- Start a session from a webhook payload
- Gate a destructive action behind human approval
- Read the full session trace in the Console

### Prerequisites

Set `ANTHROPIC_API_KEY` in your environment, then install dependencies:

In [1]:
%pip install -q "anthropic>=0.91.0" python-dotenv

In [2]:
import hashlib
import json
import os
import re
import time
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv
from utilities import wait_for_idle_status

load_dotenv()
client = Anthropic()
MODEL = os.getenv("COOKBOOK_MODEL", "claude-opus-4-6")
FIXTURE = Path("example_data/sre")

## 1. Upload a runbook skill

A [**Skill**](https://platform.claude.com/docs/en/managed-agents/skills) is a small filesystem bundle the platform mounts into the agent's context with progressive disclosure: the agent sees a one-line description up front and reads the body only when it's relevant. It's a good place for team conventions that shouldn't live in the system prompt.

The sample skill below encodes one rule — *consult the runbook before touching infrastructure* — the way a real team playbook would. You upload it once via the Skills API and reference it by ID on every agent that needs it.

In [3]:
# A real skill is usually a folder on disk (SKILL.md plus any helper
# scripts or reference docs) that you zip and upload. For this tutorial
# the SKILL.md is small enough to keep inline.
RUNBOOK_SKILL = """\
---
name: incident-runbooks
description: How to triage production incidents using the team runbooks.
---

# Incident runbooks

When an alert references a service, locate that service's recent logs
and identify the failure signature (the repeating error class, exit
code, or status pattern).

Consult the team runbooks before proposing any fix. Runbooks are
organised by failure signature — for example `oom.md`, `5xx.md`,
`latency.md`. Each one lists the triage steps for that class of
failure and the configuration that usually needs to change.

Any fix to infrastructure code must be opened as a pull request that
cites the runbook you followed. Do not patch live resources directly.
"""

skill = client.beta.skills.create(
    display_title="incident-runbooks",
    files=[("incident-runbooks/SKILL.md", RUNBOOK_SKILL.encode(), "text/markdown")],
)
print(f"skill: {skill.id} (version {skill.latest_version})")

skill: skill_01WPWHALbtEVBUWG6mHa7Tna (version 1775588716519983)


## 2. Create the agent

The agent's `tools` list combines three kinds of capability:

- [`agent_toolset_20260401`](https://platform.claude.com/docs/en/managed-agents/tools) — the built-in `bash`, `read`, `grep`, `edit`, … tools that run *inside* the sandbox. The agent uses these to investigate.
- The runbook **skill** from step 1.
- Two **custom tools** — `open_pull_request` and `request_approval` — that the agent can call but *your application* executes. The merge credential stays outside the agent's tool set, so the application executor owns the irreversible step.

The system prompt is persona and workflow only. The alert itself arrives as the first user event, so the same agent handles any incident.

In [4]:
SRE_SYSTEM_PROMPT = """\
You are an on-call SRE agent. Each user message is a PagerDuty alert
payload. Triage it to root cause and ship the minimal safe fix.

The session workspace contains the recent logs, the infrastructure
repo, and the team runbooks for the alerting service. Explore it to
find what you need.

Workflow for every alert:
1. Read the logs and identify the failure signature.
2. Find the root cause in the infrastructure repo, save a copy of the
   original file, edit it in place, then produce a unified diff with
   `diff -u`.
3. open_pull_request(title, body, diff) with the fix.
4. Use the returned pr_number and action_digest in
   request_approval(pr_number, action_digest, summary).
5. Wait for the application executor to approve or reject the exact
   action. Report its result; do not attempt the merge yourself.

Use only printable ASCII and line feeds in every custom-tool string;
write symbols such as arrows as ASCII (for example, `->`).

Keep the fix minimal — do not refactor unrelated config.
"""

agent = client.beta.agents.create(
    name="cookbook-sre-responder",
    model=MODEL,
    system=SRE_SYSTEM_PROMPT,
    skills=[{"type": "custom", "skill_id": skill.id, "version": skill.latest_version}],
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        },
        {
            "type": "custom",
            "name": "open_pull_request",
            "description": (
                "Open a pull request against the infra repo with the proposed fix. "
                "All strings must use printable ASCII and line feeds only."
            ),
            "input_schema": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "title": {"type": "string", "minLength": 1, "maxLength": 200},
                    "body": {"type": "string", "maxLength": 4000},
                    "diff": {
                        "type": "string",
                        "minLength": 1,
                        "maxLength": 12000,
                        "description": "Unified diff of the change.",
                    },
                },
                "required": ["title", "body", "diff"],
            },
        },
        {
            "type": "custom",
            "name": "request_approval",
            "description": (
                "Ask the on-call human to approve the proposed PR before merging. "
                "The summary must use printable ASCII and line feeds only."
            ),
            "input_schema": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "pr_number": {"type": "integer", "minimum": 1},
                    "action_digest": {
                        "type": "string",
                        "pattern": "^[0-9a-f]{64}$",
                    },
                    "summary": {"type": "string", "maxLength": 2000},
                },
                "required": ["pr_number", "action_digest", "summary"],
            },
        },
    ],
)
print(f"agent: {agent.id} v{agent.version}")

agent: agent_011CZpw3Y76Vu4t2j2QEosVa v1


## 3. Create an environment and mount the data

The agent needs three things in its workspace to investigate: the recent service logs, the infrastructure repo, and the team runbooks. Upload each via the Files API and list them as `resources` so they're mounted into every session at the paths the system prompt expects. A `limited`-networking cloud environment is enough because the agent only needs its own filesystem.

To keep this notebook runnable with only `ANTHROPIC_API_KEY`, the infra "repo" is a single manifest with a too-low `memory: 128Mi` limit. In production you'd replace that upload with a `github_repository` resource that clones the real repo straight into the sandbox:

```python
{
    "type": "github_repository",
    "url": "https://github.com/your-org/infra",
    "authorization_token": os.environ["GITHUB_TOKEN"],
    "checkout": {"type": "branch", "name": "main"},
    "mount_path": "infra",
}
```

In [5]:
env = client.beta.environments.create(
    name="cookbook-sre-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)


def upload(path: Path, mime: str) -> str:
    with path.open("rb") as f:
        return client.beta.files.upload(file=(path.name, f, mime)).id


log_id = upload(FIXTURE / "logs/checkout-svc.log", "text/plain")
manifest_id = upload(FIXTURE / "infra/k8s/checkout-deploy.yaml", "text/yaml")
runbook_id = upload(FIXTURE / "runbooks/oom.md", "text/markdown")

RESOURCES = [
    {"type": "file", "file_id": log_id, "mount_path": "logs/checkout-svc.log"},
    {"type": "file", "file_id": manifest_id, "mount_path": "infra/k8s/checkout-deploy.yaml"},
    {"type": "file", "file_id": runbook_id, "mount_path": "runbooks/oom.md"},
]
print(f"environment: {env.id}")

environment: env_01R6hmJkd6BhpPotXnoC7rqU


## 4. Handle the incident alert

The handler below is the one function you'd deploy — a Flask or FastAPI route that your alerting system calls when an incident fires. It creates a session referencing the agent and environment, mounts the data, and sends the alert JSON as the first `user.message` event. This example uses a [PagerDuty V3 webhook](https://developer.pagerduty.com/docs/webhooks-overview) payload, but any pager that can POST JSON works the same way; here you call the handler directly with the fixture.

In [6]:
def handle_pagerduty_webhook(payload: dict) -> str:
    incident = payload["event"]["data"]
    session = client.beta.sessions.create(
        environment_id=env.id,
        agent={"type": "agent", "id": agent.id, "version": agent.version},
        resources=RESOURCES,
        title=f"[{incident['service']['summary']}] {incident['title']}",
    )
    client.beta.sessions.events.send(
        session.id,
        events=[
            {
                "type": "user.message",
                "content": [{"type": "text", "text": json.dumps(payload, indent=2)}],
            }
        ],
    )
    return session.id


with (FIXTURE / "alert.json").open() as f:
    alert = json.load(f)

session_id = handle_pagerduty_webhook(alert)
print(f"session: {session_id}")

session: sesn_011CZpw3gtC691y7qmaLNmLM


## 5. Service the agent's custom tool calls

This is where the built-in tools and your custom tools come together. The agent's `read`/`bash`/`edit` calls run on the container and appear in the event log as `agent.tool_use` — that's the investigation, and you just print it. But when the agent calls one of your custom tools, the session goes `idle` with `stop_reason.type == "requires_action"` and waits for *your application* to respond with a `user.custom_tool_result`.

The loop below polls `events.list` and answers `open_pull_request` by writing to a local list — that's the GitHub mock — but **returns** when `request_approval` arrives. The application records the exact PR digest before presenting the decision to a human. The agent never receives a merge tool.

In production, "needs a human" usually means *post it to Slack*: render every field covered by the action digest from application-owned state, including the repository, PR, immutable head commit, base branch, merge method, title, body, and proposed diff, then attach **Approve** and **Reject** buttons. This example admits only printable ASCII and line feeds to the approval path and refuses any other code point with its field and `U+` value. The renderer also exposes non-ASCII code points and literal backslashes if validation is bypassed, so confusables, combining marks, alternate spaces and line separators, and strong RTL text cannot rely on font or layout behavior. Keep title, body, diff, and agent summary in separate non-Markdown Block Kit elements, authorize the clicking user, and bind the click to the exact channel message that was recorded as delivered. The agent's summary is untrusted context, never the authoritative description of what the human approved. The [`slack_data_bot` cookbook](slack_data_bot.ipynb) shows the Bolt wiring; here you'll approve inline in the next cell so the notebook stays self-contained.

In [7]:
prs: list[dict] = []
pending_approvals: dict[str, dict] = {}
action_reservations: dict[str, str] = {}
seen_events_by_session: dict[str, set[str]] = {}
custom_calls_by_session: dict[str, dict[str, object]] = {}
responded_events_by_session: dict[str, set[str]] = {}
DELIVERY_SLA_SECONDS = 90
MAX_APPROVAL_DIFF_CHARS = 12000
MAX_AGENT_SUMMARY_CHARS = 2000
ALERTABLE_APPROVAL_OUTCOMES = frozenset(
    {"approval_never_delivered", "expired_undecided", "expired_clicked_late"}
)
NON_REPORTABLE_CLICK_REASONS = frozenset(
    {
        "approver_identity_required",
        "approver_not_authorized",
        "approval_message_mismatch",
        "approval_not_delivered",
        "already_consumed",
    }
)
MOCK_REPOSITORY = "example/infra"
MOCK_BASE_REF = "main"
MOCK_MERGE_METHOD = "merge"


def merge_action(pr: dict) -> dict:
    content = json.dumps(
        {"title": pr["title"], "body": pr["body"], "diff": pr["diff"]},
        sort_keys=True,
        separators=(",", ":"),
    )
    return {
        "action": "github.pull_request.merge",
        "repository": pr.get("repository", MOCK_REPOSITORY),
        "pr_number": pr["number"],
        "content_hash": hashlib.sha256(content.encode()).hexdigest(),
        "base_ref": pr.get("base_ref", MOCK_BASE_REF),
        "merge_method": pr.get("merge_method", MOCK_MERGE_METHOD),
        "title": pr["title"],
        "body": pr["body"],
        "diff": pr["diff"],
    }


def digest_action(action: dict) -> str:
    canonical = json.dumps(action, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode()).hexdigest()


def merge_action_digest(pr: dict) -> str:
    return digest_action(merge_action(pr))


def _escape_review_text(text: str) -> str:
    """Render exact text without Unicode lookalikes or layout ambiguity."""
    escaped = []
    for char in str(text):
        codepoint = ord(char)
        if char == "\\":
            escaped.append("\\\\")
        elif char == "\n" or 0x20 <= codepoint <= 0x7E:
            escaped.append(char)
        elif codepoint <= 0xFFFF:
            escaped.append(f"\\u{codepoint:04x}")
        else:
            escaped.append(f"\\U{codepoint:08x}")
    return "".join(escaped)


def _review_text_violation(field: str, text: str) -> dict | None:
    """Require line feeds and printable ASCII on the approval path."""
    for char in text:
        codepoint = ord(char)
        if char == "\n" or 0x20 <= codepoint <= 0x7E:
            continue
        return {
            "status": "refused",
            "reason": "ambiguous_review_text",
            "field": field,
            "codepoint": f"U+{codepoint:04X}",
        }
    return None


def _quote_untrusted(text: str) -> str:
    """Keep agent prose inside one blockquote, not a second review block."""
    safe_text = _escape_review_text(text)
    quoted = "\n".join(f"> {line}" for line in safe_text.splitlines())
    return (quoted or "> (no summary)").replace("`", "'")


def _fence_for(text: str) -> str:
    """Return a Markdown fence longer than any backtick run in text."""
    longest = max((len(run) for run in re.findall(r"`+", text)), default=0)
    return "`" * max(3, longest + 1)


def _inline_literal(value: object) -> str:
    """Escape controls, direction marks, and backticks in inline labels."""
    escaped = str(value).encode("unicode_escape").decode("ascii")
    return escaped.replace("`", r"\x60")


def render_approval_prompt(action: dict, digest: str, agent_summary: str) -> str:
    """Render one review block while confining all agent-controlled text."""
    display_title = _escape_review_text(action["title"])
    display_body = _escape_review_text(action["body"]) or "(empty)"
    display_diff = _escape_review_text(action["diff"])
    fence = _fence_for("\n".join((display_title, display_body, display_diff)))
    return (
        "*Exact merge approval*\n"
        f"Action: `{_inline_literal(action['action'])}`\n"
        f"Repository: `{_inline_literal(action['repository'])}`\n"
        f"PR: `#{_inline_literal(action['pr_number'])}` into "
        f"`{_inline_literal(action['base_ref'])}` "
        f"using `{_inline_literal(action['merge_method'])}`\n"
        f"Content hash: `{_inline_literal(action['content_hash'])}`\n"
        f"Action digest: `{_inline_literal(digest)}`\n\n"
        "PR title:\n"
        f"{fence}text\n{display_title}\n{fence}\n\n"
        "PR body:\n"
        f"{fence}text\n{display_body}\n{fence}\n\n"
        "Proposed diff:\n"
        f"{fence}diff\n{display_diff}\n{fence}\n\n"
        "Agent context (untrusted):\n"
        f"{_quote_untrusted(agent_summary)}"
    )


def find_pr(pr_number: object) -> dict | None:
    """Resolve only a positive application-issued PR number."""
    if type(pr_number) is not int or pr_number < 1:
        return None
    return next((pr for pr in prs if pr["number"] == pr_number), None)


def validate_approval_request(args: object) -> dict | None:
    """Validate the full agent-supplied approval request before lookup."""
    expected = {"pr_number", "action_digest", "summary"}
    if not isinstance(args, dict) or set(args) != expected:
        return {"status": "refused", "reason": "invalid_tool_arguments"}
    if type(args["pr_number"]) is not int or args["pr_number"] < 1:
        return {"status": "refused", "reason": "invalid_tool_arguments"}
    if not isinstance(args["action_digest"], str) or not re.fullmatch(
        r"[0-9a-f]{64}", args["action_digest"]
    ):
        return {"status": "refused", "reason": "invalid_tool_arguments"}
    if not isinstance(args["summary"], str) or len(args["summary"]) > MAX_AGENT_SUMMARY_CHARS:
        return {"status": "refused", "reason": "invalid_tool_arguments"}
    return _review_text_violation("summary", args["summary"])


def handle_custom_tool(name: str, args: object) -> dict:
    if name == "open_pull_request":
        expected = {"title", "body", "diff"}
        if (
            not isinstance(args, dict)
            or set(args) != expected
            or not all(isinstance(args[key], str) for key in expected)
        ):
            return {"status": "refused", "reason": "invalid_tool_arguments"}
        if not args["title"] or not args["diff"]:
            return {"status": "refused", "reason": "invalid_tool_arguments"}
        if (
            len(args["title"]) > 200
            or len(args["body"]) > 4000
            or len(args["diff"]) > MAX_APPROVAL_DIFF_CHARS
        ):
            return {"status": "refused", "reason": "tool_arguments_too_large"}
        for field in ("title", "body", "diff"):
            violation = _review_text_violation(field, args[field])
            if violation is not None:
                return violation
        n = len(prs) + 1
        pr = {
            "number": n,
            "merged": False,
            "repository": MOCK_REPOSITORY,
            "base_ref": MOCK_BASE_REF,
            "merge_method": MOCK_MERGE_METHOD,
            "title": args["title"],
            "body": args["body"],
            "diff": args["diff"],
        }
        pr["action_digest"] = merge_action_digest(pr)
        prs.append(pr)
        print(f"\n── PR #{n}: {args['title']} ──")
        return {
            "pr_number": n,
            "url": f"mock://infra/pull/{n}",
            "action_digest": pr["action_digest"],
        }
    return {"status": "refused", "reason": "unknown_tool"}


def release_action_reservation(approval: dict, now: float | None = None) -> bool:
    """Release only the reservation still owned by this terminal request."""
    action_digest = approval["action_digest"]
    if action_reservations.get(action_digest) != approval["event_id"]:
        return False
    del action_reservations[action_digest]
    approval["reservation_released_at"] = time.time() if now is None else now
    return True


def register_approval(
    event_id: str,
    pr: dict,
    summary: str,
    ttl: float = 900,
    delivery_sla: float = DELIVERY_SLA_SECONDS,
    *,
    session_id: str | None = None,
) -> dict:
    """Reserve one approval request for one exact action digest."""
    authoritative_pr = find_pr(pr.get("number"))
    if authoritative_pr is not pr:
        return {"status": "refused", "reason": "unknown_pr"}
    if not isinstance(summary, str):
        return {"status": "refused", "reason": "invalid_agent_summary"}
    if len(pr["diff"]) > MAX_APPROVAL_DIFF_CHARS:
        return {"status": "refused", "reason": "approval_diff_too_large"}
    if len(summary) > MAX_AGENT_SUMMARY_CHARS:
        return {"status": "refused", "reason": "agent_summary_too_large"}
    action = merge_action(pr)
    review_fields = {
        field: action[field]
        for field in ("action", "repository", "base_ref", "merge_method", "title", "body", "diff")
    }
    review_fields["summary"] = summary
    for field, value in review_fields.items():
        violation = _review_text_violation(field, value)
        if violation is not None:
            return violation
    action_digest = digest_action(action)
    if event_id in pending_approvals:
        return {"status": "refused", "reason": "duplicate_event_id"}
    existing_event_id = action_reservations.get(action_digest)
    if existing_event_id is not None:
        return {
            "status": "refused",
            "reason": "action_already_reserved",
            "existing_event_id": existing_event_id,
        }
    requested_at = time.time()
    pending_approvals[event_id] = {
        "event_id": event_id,
        "session_id": session_id,
        "pr_number": pr["number"],
        "action_digest": action_digest,
        "action": action,
        "agent_summary": summary,
        "approval_prompt": render_approval_prompt(action, action_digest, summary),
        # Audit label for the exact delivered presentation, not an authorization check.
        "delivered_presentation": None,
        "presentation_digest": None,
        "requested_at": requested_at,
        "delivery_deadline": requested_at + delivery_sla,
        "delivered_at": None,
        "delivery_channel": None,
        "delivery_receipt_id": None,
        "answered_at": None,
        "answered_by": None,
        "answer_channel": None,
        "answer_receipt_id": None,
        "expires_at": requested_at + ttl,
        "consumed": False,
        "outcome": None,
        "reported_outcome": None,
        "incident_reported_at": None,
        "incident_report_channel": None,
        "incident_report_receipt_id": None,
        "reservation_released_at": None,
        "rearmed_at": None,
        "rearmed_by": None,
    }
    action_reservations[action_digest] = event_id
    return {"status": "pending", "event_id": event_id, "action_digest": action_digest}


def mark_approval_delivered(
    event_id: str,
    channel: str,
    receipt_id: str,
    presentation: dict,
    delivered_at: float | None = None,
) -> dict:
    """Record the channel-accepted payload, not proof that a human read it."""
    approval = pending_approvals.get(event_id)
    if approval is None:
        return {"status": "refused", "reason": "unknown_approval"}
    if not channel or not receipt_id:
        return {"status": "refused", "reason": "delivery_receipt_missing"}
    try:
        canonical_presentation = json.dumps(
            presentation,
            sort_keys=True,
            separators=(",", ":"),
            ensure_ascii=True,
            allow_nan=False,
        )
    except (TypeError, ValueError):
        return {"status": "refused", "reason": "invalid_delivery_presentation"}
    presentation_digest = hashlib.sha256(canonical_presentation.encode()).hexdigest()
    if approval["delivered_at"] is not None:
        if approval["delivery_channel"] != channel or approval["delivery_receipt_id"] != receipt_id:
            return {"status": "refused", "reason": "delivery_receipt_mismatch"}
        if approval["presentation_digest"] != presentation_digest:
            return {"status": "refused", "reason": "delivery_presentation_mismatch"}
        return {
            "status": "delivered",
            "event_id": event_id,
            "presentation_digest": presentation_digest,
        }
    if approval["outcome"] is not None:
        return {"status": "refused", "reason": "approval_terminal"}
    delivered_at = time.time() if delivered_at is None else delivered_at
    if delivered_at > approval["delivery_deadline"]:
        approval["consumed"] = True
        approval["outcome"] = "approval_never_delivered"
        release_action_reservation(approval, delivered_at)
        return {"status": "refused", "reason": "delivery_sla_exceeded"}
    approval["delivered_at"] = delivered_at
    approval["delivery_channel"] = channel
    approval["delivery_receipt_id"] = receipt_id
    approval["delivered_presentation"] = json.loads(canonical_presentation)
    approval["presentation_digest"] = presentation_digest
    return {
        "status": "delivered",
        "event_id": event_id,
        "channel": channel,
        "receipt_id": receipt_id,
        "presentation_digest": approval["presentation_digest"],
    }


def rearm_rejected_action(
    event_id: str,
    operator_id: str,
    *,
    operator_authenticated: bool = False,
    now: float | None = None,
) -> dict:
    """Release one rejected action through an authenticated operator path."""
    approval = pending_approvals.get(event_id)
    if approval is None:
        return {"status": "refused", "reason": "unknown_approval"}
    if not operator_id:
        return {"status": "refused", "reason": "operator_identity_required"}
    if not operator_authenticated:
        return {"status": "refused", "reason": "operator_authentication_required"}
    if approval["outcome"] != "human_rejected":
        return {"status": "refused", "reason": "approval_not_rejected"}
    if approval["rearmed_at"] is not None:
        return {"status": "refused", "reason": "already_rearmed"}
    rearmed_at = time.time() if now is None else now
    if not release_action_reservation(approval, rearmed_at):
        return {"status": "refused", "reason": "action_reservation_lost"}
    approval["rearmed_at"] = rearmed_at
    approval["rearmed_by"] = operator_id
    return {
        "status": "rearmed",
        "event_id": event_id,
        "action_digest": approval["action_digest"],
        "operator_id": operator_id,
    }


def stale_approvals(now: float | None = None) -> list[dict]:
    """Return terminal incidents until alert delivery is acknowledged."""
    now = time.time() if now is None else now
    incidents = []
    for event_id, approval in pending_approvals.items():
        if (
            approval["outcome"] is None
            and approval["delivered_at"] is None
            and now > approval["delivery_deadline"]
        ):
            approval["consumed"] = True
            approval["outcome"] = "approval_never_delivered"
            release_action_reservation(approval, now)
        elif approval["outcome"] is None and now > approval["expires_at"]:
            approval["consumed"] = True
            approval["outcome"] = "expired_undecided"
            release_action_reservation(approval, now)
        if approval["outcome"] not in ALERTABLE_APPROVAL_OUTCOMES:
            continue
        if approval["reported_outcome"] == approval["outcome"]:
            continue
        incident = {
            "incident_id": hashlib.sha256(f"{event_id}:{approval['outcome']}".encode()).hexdigest(),
            "event_id": event_id,
            "pr_number": approval["pr_number"],
            "status": "attention_required",
            "reason": (
                approval["outcome"]
                if approval["outcome"].startswith("approval_")
                else f"approval_{approval['outcome']}"
            ),
            "age_seconds": round(now - approval["requested_at"], 1),
        }
        if approval["outcome"] == "approval_never_delivered":
            incident["delivery_late_by_seconds"] = round(now - approval["delivery_deadline"], 1)
        else:
            incident["expired_by_seconds"] = round(now - approval["expires_at"], 1)
        incidents.append(incident)
    return incidents


def mark_approval_incident_reported(
    event_id: str,
    outcome: str,
    channel: str,
    receipt_id: str,
    reported_at: float | None = None,
) -> dict:
    """Acknowledge incident delivery only after the alert channel accepts it."""
    approval = pending_approvals.get(event_id)
    if approval is None:
        return {"status": "refused", "reason": "unknown_approval"}
    if not channel or not receipt_id:
        return {"status": "refused", "reason": "incident_receipt_missing"}
    if outcome not in ALERTABLE_APPROVAL_OUTCOMES:
        return {"status": "refused", "reason": "incident_not_reportable"}
    if approval["outcome"] != outcome:
        return {"status": "refused", "reason": "incident_outcome_mismatch"}
    if approval["reported_outcome"] is not None:
        if (
            approval["reported_outcome"] != outcome
            or approval["incident_report_channel"] != channel
            or approval["incident_report_receipt_id"] != receipt_id
        ):
            return {"status": "refused", "reason": "incident_receipt_mismatch"}
        return {"status": "reported", "event_id": event_id, "outcome": outcome}
    approval["reported_outcome"] = outcome
    approval["incident_reported_at"] = time.time() if reported_at is None else reported_at
    approval["incident_report_channel"] = channel
    approval["incident_report_receipt_id"] = receipt_id
    return {"status": "reported", "event_id": event_id, "outcome": outcome}


def consume_approval(
    event_id: str,
    decision: str,
    *,
    approver_id: str,
    approver_authorized: bool,
    channel: str,
    receipt_id: str,
) -> dict:
    approval = pending_approvals.get(event_id)
    if approval is None:
        return {"status": "refused", "reason": "unknown_approval"}
    if decision not in {"approved", "rejected"}:
        return {"status": "refused", "reason": "invalid_decision"}
    if not approver_id:
        return {"status": "refused", "reason": "approver_identity_required"}
    if not approver_authorized:
        return {"status": "refused", "reason": "approver_not_authorized"}
    if approval["outcome"] == "approval_never_delivered":
        return {"status": "refused", "reason": "approval_never_delivered"}
    if approval["outcome"] in {"expired_undecided", "expired_clicked_late"}:
        return {"status": "refused", "reason": "expired"}
    if approval["consumed"]:
        return {"status": "refused", "reason": "already_consumed"}
    if approval["delivered_at"] is None:
        return {"status": "refused", "reason": "approval_not_delivered"}
    if channel != approval["delivery_channel"] or receipt_id != approval["delivery_receipt_id"]:
        return {"status": "refused", "reason": "approval_message_mismatch"}
    now = time.time()
    approval["consumed"] = True  # reserve before the side effect or rejection
    approval["answered_at"] = now
    approval["answered_by"] = approver_id
    approval["answer_channel"] = channel
    approval["answer_receipt_id"] = receipt_id
    if now > approval["expires_at"]:
        approval["outcome"] = "expired_clicked_late"
        release_action_reservation(approval, now)
        return {"status": "refused", "reason": "expired"}
    if decision != "approved":
        approval["outcome"] = "human_rejected"
        return {"status": "rejected", "reason": "human_rejected"}
    if action_reservations.get(approval["action_digest"]) != event_id:
        approval["outcome"] = "action_reservation_lost"
        return {"status": "refused", "reason": "action_already_reserved"}
    pr = find_pr(approval["pr_number"])
    if pr is None:
        approval["outcome"] = "action_unavailable"
        release_action_reservation(approval, now)
        return {"status": "refused", "reason": "action_unavailable"}
    if merge_action_digest(pr) != approval["action_digest"]:
        approval["outcome"] = "action_changed"
        release_action_reservation(approval, now)
        return {"status": "refused", "reason": "action_changed"}
    if pr["merged"]:
        approval["outcome"] = "action_already_executed"
        return {"status": "refused", "reason": "action_already_executed"}
    pr["merged"] = True  # the application executor performs the effect
    approval["outcome"] = "executed"
    return {
        "status": "executed",
        "receipt_id": event_id,
        "pr_number": pr["number"],
        "action_digest": approval["action_digest"],
        "presentation_digest": approval["presentation_digest"],
        "delivery_receipt_id": approval["delivery_receipt_id"],
        "approver_id": approval["answered_by"],
        "approval_channel": approval["answer_channel"],
        "approval_receipt_id": approval["answer_receipt_id"],
    }


def should_send_approval_result(result: dict) -> bool:
    """Send only terminal click outcomes back to the waiting agent."""
    return result.get("reason") not in NON_REPORTABLE_CLICK_REASONS


def mark_custom_tool_responded(session_id: str, event_id: str) -> None:
    """Retain an externally delivered tool result across polling calls."""
    responded_events_by_session.setdefault(session_id, set()).add(event_id)


def run_until_approval_or_end(session_id: str) -> str | None:
    """Poll the session's event log, servicing custom tools, until either
    a request_approval call arrives (return its event_id so the caller
    can respond) or the agent ends its turn (return None)."""
    custom_calls = custom_calls_by_session.setdefault(session_id, {})
    responded = responded_events_by_session.setdefault(session_id, set())
    seen_events = seen_events_by_session.setdefault(session_id, set())
    while True:
        idle_stop = None
        for ev in client.beta.sessions.events.list(session_id):
            if ev.type == "session.status_idle":
                idle_stop = ev.stop_reason
                continue
            if ev.type == "session.status_terminated":
                return None
            if ev.id in seen_events:
                continue
            seen_events.add(ev.id)
            if ev.type == "agent.message":
                for block in ev.content:
                    if block.type == "text":
                        print(block.text, end="")
            elif ev.type == "agent.tool_use":
                print(f"\n  [{ev.name}]")
            elif ev.type == "agent.custom_tool_use":
                custom_calls[ev.id] = ev
                print(f"\n→ {ev.name}")
        if idle_stop is None:
            time.sleep(1.0)
            continue
        if idle_stop.type == "end_turn":
            return None
        if idle_stop.type == "requires_action":
            for event_id in idle_stop.event_ids:
                if event_id in responded:
                    continue
                call = custom_calls.get(event_id)
                if call is None:
                    continue
                if call.name == "request_approval":
                    validation = validate_approval_request(call.input)
                    if validation is not None:
                        client.beta.sessions.events.send(
                            session_id,
                            events=[
                                {
                                    "type": "user.custom_tool_result",
                                    "custom_tool_use_id": event_id,
                                    "content": [{"type": "text", "text": json.dumps(validation)}],
                                }
                            ],
                        )
                        responded.add(event_id)
                        continue
                    pr = find_pr(call.input.get("pr_number"))
                    if pr is None:
                        result = {"status": "refused", "reason": "unknown_pr"}
                        client.beta.sessions.events.send(
                            session_id,
                            events=[
                                {
                                    "type": "user.custom_tool_result",
                                    "custom_tool_use_id": event_id,
                                    "content": [{"type": "text", "text": json.dumps(result)}],
                                }
                            ],
                        )
                        responded.add(event_id)
                        continue
                    if call.input["action_digest"] != merge_action_digest(pr):
                        result = {"status": "refused", "reason": "action_digest_mismatch"}
                        client.beta.sessions.events.send(
                            session_id,
                            events=[
                                {
                                    "type": "user.custom_tool_result",
                                    "custom_tool_use_id": event_id,
                                    "content": [{"type": "text", "text": json.dumps(result)}],
                                }
                            ],
                        )
                        responded.add(event_id)
                        continue
                    registration = register_approval(
                        event_id, pr, call.input["summary"], session_id=session_id
                    )
                    if registration["status"] != "pending":
                        client.beta.sessions.events.send(
                            session_id,
                            events=[
                                {
                                    "type": "user.custom_tool_result",
                                    "custom_tool_use_id": event_id,
                                    "content": [{"type": "text", "text": json.dumps(registration)}],
                                }
                            ],
                        )
                        responded.add(event_id)
                        continue
                    return event_id
                result = handle_custom_tool(call.name, call.input)
                client.beta.sessions.events.send(
                    session_id,
                    events=[
                        {
                            "type": "user.custom_tool_result",
                            "custom_tool_use_id": event_id,
                            "content": [{"type": "text", "text": json.dumps(result)}],
                        }
                    ],
                )
                responded.add(event_id)
        time.sleep(1.0)


approval_event_id = run_until_approval_or_end(session_id)

I'll help you triage this production incident. The alert shows checkout-svc pods are crash-looping with 7 restarts in 5 minutes. Let me start by exploring the workspace to find logs and the infrastructure configuration.
  [bash]
Let me check the session directory for logs and infrastructure files:
  [bash]
Perfect! Let me examine the logs first to identify the failure signature:
  [read]
Perfect! I've identified the failure signature. The logs show a clear **OutOfMemoryError** pattern:

**Failure Signature:**
- Service starts and warms pricing cache (14,092 entries)
- Heap memory grows rapidly: 101MB → 118MB → 121MB (against 128MB limit)
- `pricing.recompute` function attempts allocation during garbage collection pause (412ms GC pause observed)
- Container gets OOMKilled (exit 137) after ~2 minutes
- Service restarts and immediately repeats the cycle

Now let me examine the deployment configuration to identify the root cause:
  [read]
**Root Cause Analysis:**

The deployment has memory

The agent has read the logs, matched the `OOMKilled` signature to `runbooks/oom.md` via the skill, found the 128Mi memory limit in `infra/k8s/checkout-deploy.yaml`, edited it, opened a PR, and is now waiting on you. This is the message that would land in your `#oncall` Slack channel:

In [8]:
pr = prs[0]
print(pending_approvals[approval_event_id]["approval_prompt"])
notebook_presentation = {"text": pending_approvals[approval_event_id]["approval_prompt"]}
delivery = mark_approval_delivered(
    approval_event_id,
    "notebook",
    "cell:approval-prompt",
    notebook_presentation,
)
assert delivery["status"] == "delivered"

*Exact merge approval*
Action: `github.pull_request.merge`
Repository: `example/infra`
PR: `#1` into `main` using `merge`
Content hash: `1ae610abb79a2268906ca71f19667c7e7d19128776e248f3caf7738706e6bda9`
Action digest: `b79a224cbb0eddc1526da1716724da0ec2616a7f4a4ec2e69bcf668266354587`

PR title:
```text
Fix checkout-svc OOMKilled crash-loop by increasing memory limits
```

PR body:
```text
## Issue
checkout-svc pods are in a CrashLoopBackOff state due to OutOfMemoryError. The service consistently crashes after ~2 minutes with 7 restarts in the last 5 minutes.

## Root Cause
The deployment had memory limits set to 128Mi, which is insufficient for the pricing cache operation:
- Pricing cache warms with 14,092 entries during startup
- Heap pressure builds to 118-121MB (92-94% of limit) within 90 seconds
- pricing.recompute fails to allocate 8MB, causing OOMKilled (exit 137)
- Service restarts and repeats the cycle

## Fix
Increase memory allocation to provide adequate headroom:
- **Memory 

### Exercise the executor boundary

These key-free checks run the approval logic directly. They demonstrate application-field confinement, visibility of every digested field, forged-review-block confinement, shared title/body/diff fence containment, fail-closed review-text admission plus unambiguous fallback encoding, polling state across the approval return boundary, approver authorization, exact delivered-message binding, click-before-delivery refusal without consuming the request, event-ID collision refusal, action-level duplicate refusal, malformed-decision refusal, token replay refusal, approve-A/execute-B refusal, conflicting delivery-receipt and presentation refusal, delivery failure, late approval and rejection, authenticated operator re-arm, action disappearance, and an unanswered request. The Unicode cases cover Cyrillic confusables, 96 combining marks, `U+2028`, `U+00A0`, strong RTL letters, zero-width and bidi controls, and tabs. Each refusal names the first offending code point. The application labels the mock's derived value as a content hash rather than a Git head, renders title, body, and diff inside a fence none can close, and quotes every line of the agent's summary as untrusted context. Delivery has a 90-second service-level deadline separate from the 15-minute human decision window. Undelivered and expired requests become terminal, remain reportable until an alert channel acknowledges delivery, and release their action reservation so a new request can be created; the old token stays dead. A timely human rejection keeps the action reserved so an agent cannot keep asking. Only an authenticated operator can re-arm that exact action, and execution still requires a fresh approval.

In [9]:
def demo_presentation(event_id: str) -> dict:
    return {"text": pending_approvals[event_id]["approval_prompt"]}


def demo_request(
    label: str,
    diff: str,
    ttl: float = 900,
    delivery_sla: float = DELIVERY_SLA_SECONDS,
    delivered: bool = True,
) -> tuple[str, dict]:
    pr = {
        "number": len(prs) + 1,
        "merged": False,
        "title": f"demo: {label}",
        "body": "executor-bound approval check",
        "diff": diff,
    }
    pr["action_digest"] = merge_action_digest(pr)
    prs.append(pr)
    event_id = f"demo-{label}"
    registration = register_approval(
        event_id, pr, label, ttl, delivery_sla, session_id="demo-session"
    )
    assert registration["status"] == "pending"
    if delivered:
        delivery = mark_approval_delivered(
            event_id, "test", f"test:{event_id}", demo_presentation(event_id)
        )
        assert delivery["status"] == "delivered"
    return event_id, pr


def demo_decide(
    event_id: str,
    decision: str,
    *,
    approver_id: str = "user:oncall",
    approver_authorized: bool = True,
    channel: str = "test",
    receipt_id: str | None = None,
) -> dict:
    return consume_approval(
        event_id,
        decision,
        approver_id=approver_id,
        approver_authorized=approver_authorized,
        channel=channel,
        receipt_id=receipt_id or f"test:{event_id}",
    )


results = {}

pr_count_before_injection = len(prs)
results["application_owned_pr_fields"] = handle_custom_tool(
    "open_pull_request",
    {
        "title": "safe",
        "body": "safe",
        "diff": "- a\n+ b",
        "number": 999,
        "merged": True,
        "repository": "attacker/repo",
    },
)
assert results["application_owned_pr_fields"] == {
    "status": "refused",
    "reason": "invalid_tool_arguments",
}
assert len(prs) == pr_count_before_injection
assert handle_custom_tool("open_pull_request", None) == {
    "status": "refused",
    "reason": "invalid_tool_arguments",
}
assert handle_custom_tool("unknown_tool", {}) == {
    "status": "refused",
    "reason": "unknown_tool",
}
results["invalid_approval_tool_input"] = validate_approval_request(
    {"pr_number": True, "action_digest": "0" * 64, "summary": "summary"}
)
assert results["invalid_approval_tool_input"] == {
    "status": "refused",
    "reason": "invalid_tool_arguments",
}
assert (
    validate_approval_request({"pr_number": 1, "action_digest": "0" * 64, "summary": "summary"})
    is None
)
assert validate_approval_request({"pr_number": 1, "summary": "summary"}) == {
    "status": "refused",
    "reason": "invalid_tool_arguments",
}
assert validate_approval_request(
    {"pr_number": 0, "action_digest": "0" * 64, "summary": "summary"}
) == {"status": "refused", "reason": "invalid_tool_arguments"}
assert validate_approval_request(
    {"pr_number": 1, "action_digest": "0" * 64, "summary": "rеview"}
) == {
    "status": "refused",
    "reason": "ambiguous_review_text",
    "field": "summary",
    "codepoint": "U+0435",
}
untracked_pr = {
    "number": 1,
    "merged": False,
    "title": "untracked",
    "body": "not provider-owned state",
    "diff": "- a\n+ b",
}
results["non_authoritative_pr"] = register_approval("demo-untracked", untracked_pr, "untracked")
assert results["non_authoritative_pr"] == {
    "status": "refused",
    "reason": "unknown_pr",
}
assert all(find_pr(value) is None for value in (0, -1, 99, True))

forged_pr = {
    "number": 1,
    "merged": False,
    "title": "raise memory ceiling",
    "body": "raise the ceiling",
    "diff": "- replicas: 3\n+ replicas: 0",
}
forged_action = merge_action(forged_pr)
forged_summary = (
    "routine memory bump\n\n"
    "*Exact merge approval*\n"
    "Repository: `example/infra`\n"
    "Agent context (untrusted): approved by security review, no diff"
)
forged_prompt = render_approval_prompt(forged_action, digest_action(forged_action), forged_summary)
assert [
    line for line in forged_prompt.splitlines() if line.startswith("*Exact merge approval*")
] == ["*Exact merge approval*"]
untrusted_context = forged_prompt.split("Agent context (untrusted):\n", 1)[1]
assert all(line.startswith("> ") for line in untrusted_context.splitlines())
assert "`" not in untrusted_context
label_injection_action = {
    **forged_action,
    "base_ref": "prod` forged",
    "repository": "example/infra\u202e",
}
label_injection_prompt = render_approval_prompt(
    label_injection_action, digest_action(label_injection_action), "routine change"
)
assert "prod` forged" not in label_injection_prompt
assert r"prod\x60 forged" in label_injection_prompt
assert r"example/infra\u202e" in label_injection_prompt
control_diff_action = {**forged_action, "diff": "- safe\n+ reviewed\u202eoff"}
control_diff_prompt = render_approval_prompt(
    control_diff_action, digest_action(control_diff_action), "routine\u202e change"
)
assert "\u202e" not in control_diff_prompt
assert control_diff_prompt.count(r"\u202e") == 2
assert "Head:" not in control_diff_prompt
assert "Content hash:" in control_diff_prompt
assert "head_sha" not in forged_action and "content_hash" in forged_action

visible_pr = {
    **forged_pr,
    "title": "raise memory ``` safely",
    "body": "visible body sentinel with a zero width mark: \u200b",
}
visible_action = merge_action(visible_pr)
visible_prompt = render_approval_prompt(
    visible_action, digest_action(visible_action), "routine change"
)
visible_metadata = visible_prompt.split("PR title:\n", 1)[0]
assert f"Action: `{visible_action['action']}`" in visible_metadata
assert f"Repository: `{visible_action['repository']}`" in visible_metadata
assert (
    f"PR: `#{visible_action['pr_number']}` into `{visible_action['base_ref']}` "
    f"using `{visible_action['merge_method']}`"
) in visible_metadata
assert f"Content hash: `{visible_action['content_hash']}`" in visible_metadata
assert f"Action digest: `{digest_action(visible_action)}`" in visible_metadata
display_title = _escape_review_text(visible_action["title"])
display_body = _escape_review_text(visible_action["body"])
display_diff = _escape_review_text(visible_action["diff"])
visible_fence = _fence_for("\n".join((display_title, display_body, display_diff)))
assert f"PR title:\n{visible_fence}text\n{display_title}\n{visible_fence}" in visible_prompt
assert f"PR body:\n{visible_fence}text\n{display_body}\n{visible_fence}" in visible_prompt
assert f"Proposed diff:\n{visible_fence}diff\n{display_diff}\n{visible_fence}" in visible_prompt
assert r"\u200b" in visible_prompt and "\u200b" not in visible_prompt
title_variant_action = merge_action({**visible_pr, "title": visible_pr["title"] + " "})
title_variant_prompt = render_approval_prompt(
    title_variant_action, digest_action(title_variant_action), "routine change"
)
assert digest_action(visible_action) != digest_action(title_variant_action)
assert (
    display_title in visible_prompt
    and _escape_review_text(title_variant_action["title"]) in title_variant_prompt
)

zalgo_marks = "͓͔͕͖͐͑͒͗" * 12
unicode_review_cases = {
    "confusable": (
        "audit_sink: https://аudit.intеrnаl.ехаmрlе.соm/ingest",
        r"\u0430udit",
        "U+0430",
    ),
    "zalgo": (f"+ memory: 768Mi{zalgo_marks}", r"\u0350", "U+0350"),
    "line_separator": (
        "- memory: 512Mi\u2028+ memory: 768Mi\u2028+ privileged: true",
        r"\u2028",
        "U+2028",
    ),
    "nbsp": ("- memory: 512Mi\n+\u00a0memory: 768Mi", r"\u00a0", "U+00A0"),
    "strong_rtl": (
        "+ memory: 768Mi  # עדכון 0 ,replicas",
        r"\u05e2",
        "U+05E2",
    ),
}
for case_name, (hostile_diff, expected_escape, expected_codepoint) in unicode_review_cases.items():
    expected_refusal = {
        "status": "refused",
        "reason": "ambiguous_review_text",
        "field": "diff",
        "codepoint": expected_codepoint,
    }
    assert _review_text_violation("diff", hostile_diff) == expected_refusal
    pr_count_before_unicode = len(prs)
    assert (
        handle_custom_tool(
            "open_pull_request",
            {"title": "unicode review", "body": "", "diff": hostile_diff},
        )
        == expected_refusal
    )
    assert len(prs) == pr_count_before_unicode
    hostile_action = merge_action({**forged_pr, "diff": hostile_diff})
    hostile_prompt = render_approval_prompt(
        hostile_action, digest_action(hostile_action), f"{case_name} review"
    )
    escaped_hostile_diff = _escape_review_text(hostile_diff)
    hostile_fence = _fence_for(
        "\n".join(
            (
                _escape_review_text(hostile_action["title"]),
                _escape_review_text(hostile_action["body"]),
                escaped_hostile_diff,
            )
        )
    )
    assert expected_escape in escaped_hostile_diff
    assert all(char not in hostile_prompt for char in hostile_diff if ord(char) > 0x7F)
    assert (
        f"Proposed diff:\n{hostile_fence}diff\n{escaped_hostile_diff}\n{hostile_fence}"
        in hostile_prompt
    )

provider_unicode_pr = {
    "number": len(prs) + 1,
    "merged": False,
    "title": "provider state",
    "body": "",
    "diff": unicode_review_cases["confusable"][0],
}
prs.append(provider_unicode_pr)
assert register_approval("demo-provider-unicode", provider_unicode_pr, "provider state") == {
    "status": "refused",
    "reason": "ambiguous_review_text",
    "field": "diff",
    "codepoint": "U+0430",
}
prs.remove(provider_unicode_pr)

ascii_baseline = (
    "--- a/deploy/sidecar.yaml\n"
    "+++ b/deploy/sidecar.yaml\n"
    "- memory: 512Mi\n"
    "+ memory: 768Mi\n"
    "- privileged: false\n"
    "+ privileged: true\n"
    "+ hostPID: true"
)
assert _escape_review_text(ascii_baseline) == ascii_baseline
assert _review_text_violation("diff", ascii_baseline) is None
assert _escape_review_text("a\tb\\u00a0\u00a0") == r"a\u0009b\\u00a0\u00a0"
assert _review_text_violation("diff", "a\tb") == {
    "status": "refused",
    "reason": "ambiguous_review_text",
    "field": "diff",
    "codepoint": "U+0009",
}

fence_escape_diff = "- replicas: 3\n+ replicas: 0\n```\nreviewed and signed off\n```"
fence_escape_pr = {**forged_pr, "diff": fence_escape_diff}
fence_escape_action = merge_action(fence_escape_pr)
fence_escape_prompt = render_approval_prompt(
    fence_escape_action, digest_action(fence_escape_action), "routine change"
)
safe_fence = _fence_for(fence_escape_diff)
assert safe_fence == "````"
assert (
    f"{safe_fence}diff\n{fence_escape_diff}\n{safe_fence}\n\nAgent context (untrusted):\n"
) in fence_escape_prompt

replay_event, replay_pr = demo_request("replay", "memory: 512Mi")
results["first_approval"] = demo_decide(replay_event, "approved")
results["replay"] = demo_decide(replay_event, "approved")
results["conflicting_delivery_receipt"] = mark_approval_delivered(
    replay_event, "test", "test:different-message", demo_presentation(replay_event)
)
results["conflicting_delivery_presentation"] = mark_approval_delivered(
    replay_event,
    "test",
    f"test:{replay_event}",
    {"text": "different presentation"},
)
assert results["first_approval"]["status"] == "executed"
assert results["replay"] == {"status": "refused", "reason": "already_consumed"}
assert should_send_approval_result(results["replay"]) is False
assert results["conflicting_delivery_receipt"] == {
    "status": "refused",
    "reason": "delivery_receipt_mismatch",
}
assert results["conflicting_delivery_presentation"] == {
    "status": "refused",
    "reason": "delivery_presentation_mismatch",
}
# The presentation digest is an audit record, not an authorization check.
replay_presentation_json = json.dumps(
    demo_presentation(replay_event),
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=True,
    allow_nan=False,
)
assert (
    pending_approvals[replay_event]["presentation_digest"]
    == hashlib.sha256(replay_presentation_json.encode()).hexdigest()
)
assert pending_approvals[replay_event]["delivered_presentation"] == demo_presentation(replay_event)
assert "Agent context (untrusted)" in pending_approvals[replay_event]["approval_prompt"]
assert {"repository", "content_hash", "base_ref", "merge_method"} <= set(
    pending_approvals[replay_event]["action"]
)

# A requires_action batch can contain work after the approval call. Retain it across the return.
from contextlib import redirect_stdout
from io import StringIO


class DemoEvent:
    def __init__(self, **values):
        self.__dict__.update(values)


class DemoEventsAPI:
    def __init__(self, batches):
        self.batches = list(batches)
        self.sent = []

    def list(self, session_id):
        return self.batches.pop(0)

    def send(self, session_id, events):
        self.sent.extend(events)


class DemoClient:
    def __init__(self, events_api):
        self.beta = DemoEvent(sessions=DemoEvent(events=events_api))


ordering_session = "demo-ordering-session"
had_client = "client" in globals()
original_client = globals().get("client")
original_sleep = time.sleep
try:
    with redirect_stdout(StringIO()):
        ordering_pr_result = handle_custom_tool(
            "open_pull_request",
            {"title": "ordering target", "body": "", "diff": "- a\n+ b"},
        )
        approval_call = DemoEvent(
            id="demo-ordering-approval",
            type="agent.custom_tool_use",
            name="request_approval",
            input={
                "pr_number": ordering_pr_result["pr_number"],
                "action_digest": ordering_pr_result["action_digest"],
                "summary": "approve before the next queued call",
            },
        )
        queued_call = DemoEvent(
            id="demo-ordering-queued",
            type="agent.custom_tool_use",
            name="open_pull_request",
            input={"title": "queued work", "body": "", "diff": "- x\n+ y"},
        )
        first_idle = DemoEvent(
            id="demo-ordering-idle-1",
            type="session.status_idle",
            stop_reason=DemoEvent(
                type="requires_action",
                event_ids=[approval_call.id, queued_call.id],
            ),
        )
        end_idle = DemoEvent(
            id="demo-ordering-idle-2",
            type="session.status_idle",
            stop_reason=DemoEvent(type="end_turn", event_ids=[]),
        )
        events_api = DemoEventsAPI(
            [[approval_call, queued_call, first_idle], [first_idle], [end_idle]]
        )
        client = DemoClient(events_api)
        time.sleep = lambda _: None
        assert run_until_approval_or_end(ordering_session) == approval_call.id
        assert queued_call.id in custom_calls_by_session[ordering_session]
        mark_custom_tool_responded(ordering_session, approval_call.id)
        assert run_until_approval_or_end(ordering_session) is None
        assert [event["custom_tool_use_id"] for event in events_api.sent] == [queued_call.id]
finally:
    time.sleep = original_sleep
    if had_client:
        client = original_client
    else:
        globals().pop("client", None)
results["event_state_survives_return"] = {"status": "handled"}

identity_event, identity_pr = demo_request("approver-identity", "memory: 544Mi")
results["missing_approver_identity"] = demo_decide(identity_event, "approved", approver_id="")
results["unauthorized_approver"] = demo_decide(
    identity_event, "approved", approver_authorized=False
)
results["wrong_delivery_message"] = demo_decide(
    identity_event, "approved", channel="other-channel", receipt_id="other-message"
)
assert results["missing_approver_identity"] == {
    "status": "refused",
    "reason": "approver_identity_required",
}
assert results["unauthorized_approver"] == {
    "status": "refused",
    "reason": "approver_not_authorized",
}
assert should_send_approval_result(results["unauthorized_approver"]) is False
assert results["wrong_delivery_message"] == {
    "status": "refused",
    "reason": "approval_message_mismatch",
}
assert should_send_approval_result(results["wrong_delivery_message"]) is False
assert pending_approvals[identity_event]["consumed"] is False
results["authorized_bound_approval"] = demo_decide(identity_event, "approved")
assert results["authorized_bound_approval"]["status"] == "executed"
assert should_send_approval_result(results["authorized_bound_approval"]) is True
assert results["authorized_bound_approval"]["approver_id"] == "user:oncall"
assert results["authorized_bound_approval"]["approval_receipt_id"] == (f"test:{identity_event}")
assert pending_approvals[identity_event]["answered_by"] == "user:oncall"
assert identity_pr["merged"] is True

pre_delivery_event, pre_delivery_pr = demo_request(
    "pre-delivery-click", "memory: 560Mi", delivered=False
)
results["click_before_delivery_receipt"] = demo_decide(pre_delivery_event, "approved")
assert results["click_before_delivery_receipt"] == {
    "status": "refused",
    "reason": "approval_not_delivered",
}
assert should_send_approval_result(results["click_before_delivery_receipt"]) is False
assert pending_approvals[pre_delivery_event]["consumed"] is False
assert (
    mark_approval_delivered(
        pre_delivery_event,
        "test",
        f"test:{pre_delivery_event}",
        demo_presentation(pre_delivery_event),
    )["status"]
    == "delivered"
)
results["approval_after_delivery_receipt"] = demo_decide(pre_delivery_event, "approved")
assert results["approval_after_delivery_receipt"]["status"] == "executed"
assert pre_delivery_pr["merged"] is True

invalid_event, invalid_pr = demo_request("invalid-decision", "memory: 576Mi")
results["invalid_decision"] = demo_decide(invalid_event, "approvd")
results["approval_after_invalid_decision"] = demo_decide(invalid_event, "approved")
assert results["invalid_decision"] == {
    "status": "refused",
    "reason": "invalid_decision",
}
assert results["approval_after_invalid_decision"]["status"] == "executed"
assert invalid_pr["merged"] is True

event_key, event_pr = demo_request("event-id", "memory: 704Mi")
other_pr = {
    "number": len(prs) + 1,
    "merged": False,
    "title": "demo: event-id collision",
    "body": "different action, same event identifier",
    "diff": "memory: 896Mi",
}
other_pr["action_digest"] = merge_action_digest(other_pr)
prs.append(other_pr)
results["duplicate_event_id"] = register_approval(
    event_key, other_pr, "must not overwrite the first event"
)
assert results["duplicate_event_id"] == {
    "status": "refused",
    "reason": "duplicate_event_id",
}
assert pending_approvals[event_key]["pr_number"] == event_pr["number"]
results["original_event_after_collision"] = demo_decide(event_key, "approved")
assert results["original_event_after_collision"]["status"] == "executed"

reserved_event, reserved_pr = demo_request("action-reservation", "memory: 768Mi")
results["duplicate_action_request"] = register_approval(
    "demo-action-reservation-retry", reserved_pr, "same exact merge, second token"
)
results["reserved_action_approval"] = demo_decide(reserved_event, "approved")
assert results["duplicate_action_request"] == {
    "status": "refused",
    "reason": "action_already_reserved",
    "existing_event_id": reserved_event,
}
assert results["reserved_action_approval"]["status"] == "executed"
assert reserved_pr["merged"] is True

mutation_event, mutation_pr = demo_request("mutation", "memory: 512Mi")
mutation_pr["diff"] = "memory: 512Mi\nreplicas: 0"
results["approve_A_execute_B"] = demo_decide(mutation_event, "approved")
assert results["approve_A_execute_B"] == {"status": "refused", "reason": "action_changed"}
assert mutation_pr["merged"] is False

expiry_event, expiry_pr = demo_request("expiry", "memory: 512Mi", ttl=-1)
results["late_approval"] = demo_decide(expiry_event, "approved")
results["late_click_visibility"] = next(
    item for item in stale_approvals() if item["event_id"] == expiry_event
)
assert results["late_approval"] == {"status": "refused", "reason": "expired"}
assert results["late_click_visibility"]["reason"] == "approval_expired_clicked_late"
repeat_expiry_incident = next(
    item for item in stale_approvals() if item["event_id"] == expiry_event
)
assert repeat_expiry_incident["incident_id"] == results["late_click_visibility"]["incident_id"]
results["late_click_incident_reported"] = mark_approval_incident_reported(
    expiry_event, "expired_clicked_late", "alerts", "alert:expiry"
)
assert results["late_click_incident_reported"]["status"] == "reported"
assert (
    mark_approval_incident_reported(expiry_event, "expired_clicked_late", "alerts", "alert:expiry")[
        "status"
    ]
    == "reported"
)
assert mark_approval_incident_reported(
    expiry_event, "expired_clicked_late", "alerts", "alert:different"
) == {"status": "refused", "reason": "incident_receipt_mismatch"}
assert all(item["event_id"] != expiry_event for item in stale_approvals())
results["approval_after_expiry"] = register_approval(
    "demo-expiry-retry", expiry_pr, "fresh approval after terminal expiry"
)
assert results["approval_after_expiry"]["status"] == "pending"
assert (
    mark_approval_delivered(
        "demo-expiry-retry",
        "test",
        "test:demo-expiry-retry",
        demo_presentation("demo-expiry-retry"),
    )["status"]
    == "delivered"
)
results["reauthorized_after_expiry"] = demo_decide("demo-expiry-retry", "approved")
assert results["reauthorized_after_expiry"]["status"] == "executed"
assert expiry_pr["merged"] is True

late_reject_event, late_reject_pr = demo_request("late-rejection", "memory: 608Mi", ttl=-1)
results["late_rejection"] = demo_decide(late_reject_event, "rejected")
results["late_rejection_visibility"] = next(
    item for item in stale_approvals() if item["event_id"] == late_reject_event
)
assert results["late_rejection"] == {"status": "refused", "reason": "expired"}
assert results["late_rejection_visibility"]["reason"] == ("approval_expired_clicked_late")
assert any(item["event_id"] == late_reject_event for item in stale_approvals())
results["late_rejection_incident_reported"] = mark_approval_incident_reported(
    late_reject_event, "expired_clicked_late", "alerts", "alert:late-rejection"
)
assert results["late_rejection_incident_reported"]["status"] == "reported"
assert all(item["event_id"] != late_reject_event for item in stale_approvals())
assert late_reject_pr["merged"] is False

reject_event, reject_pr = demo_request("rejection", "memory: 512Mi")
results["rejection"] = demo_decide(reject_event, "rejected")
results["approval_after_rejection"] = demo_decide(reject_event, "approved")
results["new_request_after_rejection"] = register_approval(
    "demo-rejection-retry", reject_pr, "same action after a human said no"
)
assert results["rejection"] == {"status": "rejected", "reason": "human_rejected"}
assert results["approval_after_rejection"] == {"status": "refused", "reason": "already_consumed"}
assert results["new_request_after_rejection"] == {
    "status": "refused",
    "reason": "action_already_reserved",
    "existing_event_id": reject_event,
}
assert reject_pr["merged"] is False
results["rearm_without_identity"] = rearm_rejected_action(
    reject_event, "", operator_authenticated=True
)
results["rearm_without_authentication"] = rearm_rejected_action(reject_event, "operator:sre-oncall")
results["operator_rearm"] = rearm_rejected_action(
    reject_event, "operator:sre-oncall", operator_authenticated=True
)
results["duplicate_operator_rearm"] = rearm_rejected_action(
    reject_event, "operator:sre-oncall", operator_authenticated=True
)
assert results["rearm_without_identity"] == {
    "status": "refused",
    "reason": "operator_identity_required",
}
assert results["rearm_without_authentication"] == {
    "status": "refused",
    "reason": "operator_authentication_required",
}
assert results["operator_rearm"]["status"] == "rearmed"
assert results["duplicate_operator_rearm"] == {
    "status": "refused",
    "reason": "already_rearmed",
}
assert pending_approvals[reject_event]["rearmed_by"] == "operator:sre-oncall"
assert demo_decide(reject_event, "approved") == {
    "status": "refused",
    "reason": "already_consumed",
}
results["fresh_request_after_rearm"] = register_approval(
    "demo-rejection-rearmed", reject_pr, "fresh request after operator re-arm"
)
assert results["fresh_request_after_rearm"]["status"] == "pending"
assert (
    mark_approval_delivered(
        "demo-rejection-rearmed",
        "test",
        "test:demo-rejection-rearmed",
        demo_presentation("demo-rejection-rearmed"),
    )["status"]
    == "delivered"
)
results["reauthorized_after_rearm"] = demo_decide("demo-rejection-rearmed", "approved")
assert results["reauthorized_after_rearm"]["status"] == "executed"
assert reject_pr["merged"] is True

unavailable_event, unavailable_pr = demo_request("action-unavailable", "memory: 624Mi")
prs.remove(unavailable_pr)
results["action_unavailable"] = demo_decide(unavailable_event, "approved")
assert results["action_unavailable"] == {
    "status": "refused",
    "reason": "action_unavailable",
}
assert pending_approvals[unavailable_event]["reservation_released_at"] is not None
assert unavailable_pr["merged"] is False

undelivered_event, undelivered_pr = demo_request(
    "undelivered",
    "memory: 640Mi",
    delivery_sla=90,
    delivered=False,
)
undelivered_now = pending_approvals[undelivered_event]["delivery_deadline"] + 1
results["registered_but_never_delivered"] = next(
    item for item in stale_approvals(now=undelivered_now) if item["event_id"] == undelivered_event
)
results["undelivered_token_stays_dead"] = demo_decide(undelivered_event, "approved")
results["fresh_request_after_delivery_failure"] = register_approval(
    "demo-undelivered-retry",
    undelivered_pr,
    "fresh approval after delivery failure",
)
assert results["registered_but_never_delivered"]["reason"] == ("approval_never_delivered")
assert results["undelivered_token_stays_dead"] == {
    "status": "refused",
    "reason": "approval_never_delivered",
}
assert results["fresh_request_after_delivery_failure"]["status"] == "pending"
assert (
    mark_approval_delivered(
        "demo-undelivered-retry",
        "test",
        "test:demo-undelivered-retry",
        demo_presentation("demo-undelivered-retry"),
    )["status"]
    == "delivered"
)
results["reauthorized_after_delivery_failure"] = demo_decide("demo-undelivered-retry", "approved")
assert results["reauthorized_after_delivery_failure"]["status"] == "executed"
assert any(item["event_id"] == undelivered_event for item in stale_approvals(now=undelivered_now))
results["non_delivery_incident_reported"] = mark_approval_incident_reported(
    undelivered_event,
    "approval_never_delivered",
    "alerts",
    "alert:undelivered",
)
assert results["non_delivery_incident_reported"]["status"] == "reported"
assert all(item["event_id"] != undelivered_event for item in stale_approvals(now=undelivered_now))
assert undelivered_pr["merged"] is True

silent_event, silent_pr = demo_request("silence", "memory: 512Mi", ttl=-1)
results["unanswered"] = next(item for item in stale_approvals() if item["event_id"] == silent_event)
assert results["unanswered"]["reason"] == "approval_expired_undecided"
assert any(item["event_id"] == silent_event for item in stale_approvals())
results["unanswered_incident_reported"] = mark_approval_incident_reported(
    silent_event, "expired_undecided", "alerts", "alert:unanswered"
)
assert results["unanswered_incident_reported"]["status"] == "reported"
assert all(item["event_id"] != silent_event for item in stale_approvals())
assert silent_pr["merged"] is False

for name, result in results.items():
    detail = f" {result['reason']}" if result.get("reason") else ""
    print(f"{name}: {result['status']}{detail}")

application_owned_pr_fields: refused invalid_tool_arguments
invalid_approval_tool_input: refused invalid_tool_arguments
non_authoritative_pr: refused unknown_pr
first_approval: executed
replay: refused already_consumed
conflicting_delivery_receipt: refused delivery_receipt_mismatch
conflicting_delivery_presentation: refused delivery_presentation_mismatch
event_state_survives_return: handled
missing_approver_identity: refused approver_identity_required
unauthorized_approver: refused approver_not_authorized
wrong_delivery_message: refused approval_message_mismatch
authorized_bound_approval: executed
click_before_delivery_receipt: refused approval_not_delivered
approval_after_delivery_receipt: executed
invalid_decision: refused invalid_decision
approval_after_invalid_decision: executed
duplicate_event_id: refused duplicate_event_id
original_event_after_collision: executed
duplicate_action_request: refused action_already_reserved
reserved_action_approval: executed
approve_A_execute_B: refu

## 6. Approve and let the executor merge

The application reserves both the event ID and exact PR digest when it registers the first approval request, so neither an event collision nor a retry can create a second token for the same merge. It allowlists application-owned PR fields, refuses review text outside line feeds and printable ASCII, renders every digested field in one review block, contains title, body, and diff with one dynamic fence, and retains unambiguous renderer encoding as defense in depth. It keeps queued tool-call state across the approval return boundary and records the exact delivered presentation and channel receipt only after the external post succeeds. The decision path then requires an authorized approver and the exact delivered channel receipt before it consumes the request and performs the merge. Only then does it return an execution receipt. The presentation digest in that receipt is an audit label for the channel-accepted payload, not an authorization check. Undelivered and expired requests terminate the old token and release the reservation for a fresh request. A timely human rejection keeps the action reserved until an authenticated operator re-arms that exact action; the rejected token stays dead and a fresh approval is still required. In the Slack version this runs across the message sender and button-click handler. The merge credential never enters the agent's tool set.

In [10]:
approval_receipt = consume_approval(
    approval_event_id,
    decision="approved",
    approver_id="user:notebook-oncall",
    approver_authorized=True,
    channel="notebook",
    receipt_id="cell:approval-prompt",
)
client.beta.sessions.events.send(
    session_id,
    events=[
        {
            "type": "user.custom_tool_result",
            "custom_tool_use_id": approval_event_id,
            "content": [{"type": "text", "text": json.dumps(approval_receipt)}],
        }
    ],
)
mark_custom_tool_responded(session_id, approval_event_id)

run_until_approval_or_end(session_id)
print(f"\n\nReceipt: {approval_receipt['receipt_id']}")
print(f"PR #{pr['number']} merged: {prs[0]['merged']}")

The executor verified the approval receipt and merged the exact proposed PR.
## ✅ Incident Resolved

**Summary:**
- **Status:** MERGED (PR #1)
- **Failure:** checkout-svc crash-loop with OOMKilled (exit 137)
- **Root Cause:** Memory limit of 128Mi was insufficient for pricing cache (14,092 entries)
- **Fix Applied:** 
  - Memory request: 128Mi → 256Mi
  - Memory limit: 128Mi → 512Mi

**What Happened:**
1. Service loaded 14k+ pricing cache entries during startup
2. Heap grew to 118-121MB within ~2 minutes (92-94% of 128Mi limit)
3. pricing.recompute failed to allocate 8MB, triggering OutOfMemoryError
4. Container was OOMKilled and restarted, repeating the cycle

**Expected Outcome:**
With 512Mi limit and 256Mi request, the service will have sufficient memory headroom for:
- Pricing cache operations
- Normal request processing
- JVM garbage collection pauses
- No more CrashLoopBackOff

The deployment update will trigger a rolling restart of the 3 replicas, allowing pods to spawn with the

## 7. Review the run in the Console

The important boundary is outside the prompt: Claude can propose the PR and provide context, but it cannot overwrite application-owned PR identity, define the authoritative human display, or call the merge operation. The application renders every field in the digest, labels the mock's derived value as a content hash, keeps pending event state across poller returns, reserves the event ID and action digest, applies a 90-second delivery deadline and a separate 15-minute human window, requires an authorized approver clicking the exact delivered message, consumes the approval once, performs the merge, and returns a receipt. It refuses extra tool fields, invalid PR numbers, event collisions, malformed decisions, unauthorized or cross-message clicks, conflicting delivery receipts, duplicate requests for the same action, approve-A/execute-B, expired approvals or rejections, token replay, action disappearance, and resurrection after rejection. `stale_approvals()` preserves non-delivery, unanswered expiry, and late-click expiry as separate terminal outcomes and returns them until `mark_approval_incident_reported()` records a matching channel receipt; none can trigger the merge. Expired and undelivered requests release only their own reservation after becoming terminal. The `action_reservation_lost` check remains defense-in-depth for a future durable-store race; the single-process harness does not claim to exercise it.

This pattern incorporates the production failure report shared by Anton Dziatkovskii and Mycroft in [issue #701](https://github.com/anthropics/claude-cookbooks/issues/701) and the fleet-derived negative cases they ran in [PR #803](https://github.com/anthropics/claude-cookbooks/pull/803#issuecomment-5160512049), together with the action-bound admission and consumption work contributed by [EMILIA Protocol](https://github.com/emiliaprotocol/emilia-protocol/pull/451). The collaboration is concrete: Mycroft supplies incident-shaped failure cases; EMILIA turns them into executor invariants and receipts.

Because the investigation ran as a Managed Agents session, the file reads, `bash` diff, manifest edit, proposal and approval request are persisted in the Console. The application-owned receipt records the final merge boundary separately. Open the session under **Managed Agents → Sessions** to review both sides of the handoff:

<img src="https://raw.githubusercontent.com/anthropics/claude-cookbooks/main/managed_agents/example_data/sre/console_session.png" alt="Console session view for the incident-response run" width="700" />

### Cleanup

Archive the session and the resources you created.

In [11]:
wait_for_idle_status(client, session_id)
client.beta.sessions.archive(session_id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
client.beta.skills.versions.delete(skill.latest_version, skill_id=skill.id)
client.beta.skills.delete(skill.id)
print("archived")

archived


## Next steps: production wiring

Three swaps take this from notebook to on-call.

**Approve in Slack.** When `request_approval` arrives, post it to the on-call channel with Block Kit buttons and send the `user.custom_tool_result` back from the action handler. Admit only line feeds and printable ASCII, render every digested field, and retain explicit code-point encoding as a renderer backstop. Keep trusted labels, title, body, diff, and agent prose in separate structured blocks so untrusted content is never parsed as message markup. Bolt verifies Slack request signatures; your authorization policy must still decide which verified Slack users may approve. The [`slack_data_bot` cookbook](slack_data_bot.ipynb) covers the Bolt app setup; the approval-specific wiring is below:

```python
def approval_blocks(event_id):
    approval = pending_approvals[event_id]
    action = approval["action"]
    display_title = _escape_review_text(action["title"])
    display_body = _escape_review_text(action["body"]) or "(empty)"
    display_diff = _escape_review_text(action["diff"])
    event_tag = hashlib.sha256(event_id.encode()).hexdigest()[:12]
    button_value = json.dumps({"event_id": event_id}, separators=(",", ":"))
    details = (
        f"Action: {_inline_literal(action['action'])}\n"
        f"Repository: {_inline_literal(action['repository'])}\n"
        f"PR: #{action['pr_number']} into {_inline_literal(action['base_ref'])} "
        f"using {_inline_literal(action['merge_method'])}\n"
        f"Content hash: {_inline_literal(action['content_hash'])}\n"
        f"Action digest: {_inline_literal(approval['action_digest'])}"
    )

    def preformatted_blocks(name, text):
        return [
            {
                "type": "rich_text",
                "block_id": f"approval-{name}-{event_tag}-{offset}",
                "elements": [
                    {
                        "type": "rich_text_preformatted",
                        "elements": [
                            {"type": "text", "text": text[offset : offset + 2800]}
                        ],
                    }
                ],
            }
            for offset in range(0, len(text) or 1, 2800)
        ]

    return [
        {"type": "header", "text": {"type": "plain_text", "text": "Exact merge approval"}},
        {"type": "section", "text": {"type": "plain_text", "text": details}},
        {"type": "header", "text": {"type": "plain_text", "text": "PR title"}},
        *preformatted_blocks("title", display_title),
        {"type": "header", "text": {"type": "plain_text", "text": "PR body"}},
        *preformatted_blocks("body", display_body),
        {"type": "header", "text": {"type": "plain_text", "text": "Proposed diff"}},
        *preformatted_blocks("diff", display_diff),
        {"type": "header", "text": {"type": "plain_text", "text": "Agent context (untrusted)"}},
        {
            "type": "section",
            "text": {
                "type": "plain_text",
                "text": _escape_review_text(approval["agent_summary"])
                or "(no summary)",
            },
        },
        {
            "type": "actions",
            "block_id": f"approval-actions-{event_tag}",
            "elements": [
                {"type": "button", "text": {"type": "plain_text", "text": "Approve"},
                 "action_id": "approval_approve", "value": button_value},
                {"type": "button", "text": {"type": "plain_text", "text": "Reject"},
                 "action_id": "approval_reject", "value": button_value},
            ],
        },
    ]

def post_for_approval(event_id):
    approval = pending_approvals[event_id]
    action = approval["action"]
    presentation = {
        "text": (
            f"Approval required for {_inline_literal(action['repository'])} "
            f"PR #{action['pr_number']} at content hash "
            f"{_inline_literal(action['content_hash'])}"
        ),
        "blocks": approval_blocks(event_id),
    }
    response = slack.client.chat_postMessage(
        channel=ONCALL_CHANNEL,
        **presentation,
    )
    delivery = mark_approval_delivered(
        event_id, response["channel"], response["ts"], presentation
    )
    if delivery["status"] != "delivered":
        raise RuntimeError(f"approval delivery was not admitted: {delivery}")
    return response

@slack.action("approval_approve")
@slack.action("approval_reject")
def on_approval_decision(ack, body):
    ack()
    action = body["actions"][0]
    request = json.loads(action["value"])
    event_id = request["event_id"]
    session_id = pending_approvals[event_id]["session_id"]
    if session_id is None:
        return
    approver_id = body["user"]["id"]
    container = body["container"]
    decision = "approved" if action["action_id"] == "approval_approve" else "rejected"
    approval_receipt = consume_approval(
        event_id,
        decision=decision,
        approver_id=approver_id,
        approver_authorized=approver_id in ONCALL_APPROVER_IDS,
        channel=container["channel_id"],
        receipt_id=container["message_ts"],
    )
    if not should_send_approval_result(approval_receipt):
        return
    client.beta.sessions.events.send(
        session_id,
        events=[{"type": "user.custom_tool_result",
                 "custom_tool_use_id": event_id,
                 "content": [{"type": "text",
                              "text": json.dumps(approval_receipt)}]}],
    )
    mark_custom_tool_responded(session_id, event_id)
```

In production, build the action from provider-owned values: repository identity, PR number, immutable head commit, base branch, merge method, and any provider preconditions. Do not merge tool input into those records; allowlist proposal fields and re-fetch provider state. At execution, pass the stored head commit to the provider as a conditional precondition; a separate check followed by an unconditional merge leaves a time-of-check/time-of-use gap. Persist the exact channel-accepted presentation payload and its audit-only digest alongside registered, delivery deadline, channel acceptance, channel receipt ID, answered by, answer channel and receipt, reservation released, re-armed by, re-armed at, terminal outcome, incident report channel and receipt, and last-reported outcome. The digest records the delivered payload; it does not verify that a human read it or authorize the effect. A signed Slack interaction proves who clicked, not that the user was authorized; enforce a current approver policy and bind the click to the recorded channel and message timestamp. Reserve both `event_id` and `action_digest` atomically in the durable store; persist the poller cursor, pending custom calls, and responded event IDs in session-scoped state as well. The notebook dictionaries are only a single-process mock. Keep a terminal incident reportable until the alert channel confirms delivery, use its stable `incident_id` as the channel idempotency key, then record that receipt atomically so later sweeps stay quiet. Undelivered and expired requests may release their reservation only after the old token is terminal; creating the replacement still requires a fresh approval request. A timely human rejection keeps its action reserved until an authenticated operator re-arms that exact action through an operator-only control path. The rejected token remains terminal, and the replacement requires a fresh human approval. None of these liveness paths retries the merge.

To drop the polling loop entirely, register a Console webhook on `session.requires_action` — the platform calls your endpoint the moment the agent pauses, and you post to Slack from there.

**GitHub instead of the mock.** Give the agent a GitHub credential that can read and open pull requests, but cannot merge them. Keep the merge credential in the application executor that consumes the approval. [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb) walks through per-user credentials.

```python
agent = client.beta.agents.create(
    ...,
    mcp_servers=[{"type": "url", "name": "github",
                  "url": "https://api.githubcopilot.com/mcp/"}],
    tools=[{"type": "agent_toolset_20260401"},
           {"type": "mcp_toolset", "server_name": "github"},  # no merge scope
           request_approval_tool],
)
session = client.beta.sessions.create(..., vault_ids=[github_vault.id])
```

**Live logs instead of a fixture.** Pass `DD_API_KEY` / `DD_APP_KEY` through the environment config and let the agent `curl` the Datadog Logs API from `bash` instead of reading a mounted file.

> See also the [Agent SDK site-reliability agent](https://github.com/anthropics/claude-cookbooks/blob/main/claude_agent_sdk/03_The_site_reliability_agent.ipynb) for the same problem solved with the local Agent SDK instead of the hosted Managed Agents runtime.


## What you learned

- Trigger a session from any external event — one API call from a PagerDuty webhook started the whole run.
- Attach a **Skill** to give the agent your team's conventions.
- Mount data with **resources**: `github_repository` for code, `file` for logs and runbooks.
- Use **custom tools** to call back into your app and gate actions on human approval via `requires_action`.
- Bind approval to the exact action and delivered message, authorize the approver, reserve the action across duplicate requests, expire it, consume it once, and keep the irreversible credential in the application executor.
- Track registration, delivery, approver identity, decision, terminal outcome, and incident-delivery receipt separately so silence and late clicks remain visible until reporting succeeds without weakening fail-closed behavior.
- Join the Console session trace with the application-owned execution receipt for an end-to-end audit trail.

Swap the mocks for provider-owned GitHub state, an authorized and message-bound Slack approval path, a durable transactional store, and live logs, then validate the provider's conditional-merge and uncertain-outcome behavior before production use.